<a href="https://colab.research.google.com/github/datnguyen76/thuchanhtrituenhantao/blob/main/Tuan_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài tập thực hành AI - Tuần 6

Notebook này được xây dựng theo cấu trúc tương tự chuỗi bài thực hành trước (Markdown giải thích từng bước $\rightarrow$ Code thực thi & trực quan hóa $\rightarrow$ Bảng đánh giá & nhận xét), sẵn sàng chạy trực tiếp trên Google Colab hoặc Jupyter Notebook.

---

## 0. Thiết lập môi trường & Nạp thư viện
Chạy ô code bên dưới để khởi tạo môi trường tính toán và nạp các thư viện cần thiết (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")
print("Đã nạp thư viện thành công!")

---
## 1. Khám phá dữ liệu (EDA) & Tiền xử lý
Kiểm tra chất lượng dữ liệu, thống kê mô tả, phân bố lớp và chuẩn bị tập huấn luyện/kiểm tra ổn định.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

print("Kích thước dữ liệu:", df.shape)
print("Số lượng giá trị thiếu:", df.isnull().sum().sum())
print("Số dòng trùng lặp:", df.duplicated().sum())
print("Phân bố lớp target:")
print(df["target"].value_counts().sort_index())
display(df.head())

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Tập train: {X_train.shape}, Tập test: {X_test.shape}")

---
## 2. Huấn luyện, so sánh đa mô hình & Mô hình kết hợp (Voting Ensemble)
So sánh các mô hình đơn lẻ cùng với `VotingClassifier` trên tập kiểm tra theo các chỉ số: **Accuracy, Precision, Recall, F1-Score và ROC-AUC**.

In [ ]:
base_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "KNN (k=7)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=7))
    ]),
    "SVM (RBF, probability=True)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.5, probability=True, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
}

voting_clf = VotingClassifier(
    estimators=[
        ("lr", base_models["Logistic Regression"]),
        ("svm", base_models["SVM (RBF, probability=True)"]),
        ("rf", base_models["Random Forest"]),
        ("gb", base_models["Gradient Boosting"])
    ],
    voting="soft"
)

all_models = {**base_models, "Soft Voting Ensemble": voting_clf}

rows = []
trained = {}

for name, mdl in all_models.items():
    mdl.fit(X_train, y_train)
    trained[name] = mdl
    y_pred = mdl.predict(X_test)
    y_prob = mdl.predict_proba(X_test)[:, 1]

    rows.append({
        "Mô hình": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    })

df_results = pd.DataFrame(rows).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
display(df_results)

---
## 3. Phân tích đường cong ROC, Confusion Matrix & Learning Curve
Trực quan hóa đường cong **ROC** của các mô hình tiêu biểu, ma trận nhầm lẫn của mô hình dẫn đầu và đường cong học tập (**Learning Curve**) để kiểm tra overfitting/underfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

# 1. ROC Curves
for name in ["Soft Voting Ensemble", "Gradient Boosting", "Random Forest", "Logistic Regression", "Decision Tree"]:
    probs = trained[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random Guess")
axes[0].set_title("Đường cong ROC trên tập Test")
axes[0].set_xlabel("False Positive Rate (FPR)")
axes[0].set_ylabel("True Positive Rate (TPR)")
axes[0].legend(loc="lower right", fontsize=8.5)
axes[0].grid(alpha=0.3)

# 2. Confusion Matrix cho mô hình đứng đầu bảng
top_name = df_results.loc[0, "Mô hình"]
top_pred = trained[top_name].predict(X_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, top_pred,
    display_labels=["Malignant (0)", "Benign (1)"],
    cmap="Blues", ax=axes[1]
)
axes[1].set_title(f"Confusion Matrix - {top_name}")

plt.tight_layout()
plt.show()

print(f"=== CLASSIFICATION REPORT ({top_name}) ===")
print(classification_report(y_test, top_pred, target_names=["Malignant (0)", "Benign (1)"]))

In [ ]:
# Learning Curve để chẩn đoán mức độ học & tổng quát hóa
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes, train_scores, val_scores = learning_curve(
    trained[top_name], X, y, cv=cv, scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 6), n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8.5, 4.5))
plt.plot(train_sizes, train_mean, marker="o", color="steelblue", label="Training F1")
plt.plot(train_sizes, val_mean, marker="s", color="darkorange", label="Cross-Validation F1")
plt.title(f"Learning Curve - {top_name}")
plt.xlabel("Số lượng mẫu huấn luyện")
plt.ylabel("F1-Score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Nhận xét & Tổng kết Tuần 6:
* Kết hợp nhiều mô hình đa dạng qua `Soft Voting` giúp cân bằng phương sai và tận dụng điểm mạnh của cả mô hình tuyến tính, mô hình biên/nhân và mô hình cây/boosting.
* Kết hợp `ROC-AUC`, `Confusion Matrix` và `Learning Curve` cho phép đánh giá đồng thời khả năng phân tách ngưỡng xác suất, loại lỗi cụ thể và độ ổn định khi tăng kích thước dữ liệu huấn luyện.